# Sensibilità dell'errore SDE-Net alla threshold MTGFlow

Analisi esclusivamente **post-processing** dei CSV già prodotti. Il notebook non addestra e non modifica MTGFlow o SDE-Net.

Obiettivi:
- asse X: threshold MTGFlow;
- asse Y: MAE/RMSE SDE-Net;
- curve separate per campioni normali e rari;
- numero e percentuale di campioni che cambiano classe;
- esportazione di una threshold candidata e delle soglie effettive per località.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Eseguire il notebook dalla root del repository o da notebooks/.')

MTGFLOW_SEED_DIR = Path(os.environ.get(
    'MTGFLOW_SEED_DIR', ROOT / 'outputs' / 'pvgis_mtgflow' / 'downstream_dense' / 'seed_15'
)).resolve()
MTGFLOW_TEST_CSV = MTGFLOW_SEED_DIR / 'anomaly_scores.csv'
MTGFLOW_TRAIN_CSV = MTGFLOW_SEED_DIR / 'train_anomaly_scores.csv'
SDE_PREDICTIONS_CSV = Path(os.environ.get(
    'SDE_PREDICTIONS_CSV', ROOT / 'outputs' / 'REPLACE_WITH_SDE_RUN' / 'predictions.csv'
)).resolve()
OUT_DIR = Path(os.environ.get(
    'MTGFLOW_THRESHOLD_OUT_DIR', ROOT / 'outputs' / 'mtgflow_threshold_sensitivity'
)).resolve()

# Il default varia la threshold storica specifica di ogni località tramite
# threshold_i(alpha) = alpha * threshold_i_originale. alpha=1 riproduce il CSV.
SWEEP_MODE = 'saved_threshold_multiplier'  # oppure 'absolute_score'
MULTIPLIERS = np.linspace(0.50, 2.00, 61)
ABSOLUTE_TRAIN_QUANTILES = np.linspace(0.90, 0.999, 61)
SELECTED_COORDINATE = 1.0  # modificare soltanto dopo aver studiato le curve
DAYTIME_ONLY = True
DAYTIME_THRESHOLD_WM2 = 10.0
MIN_JOIN_COVERAGE = 0.95
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('MTGFlow test :', MTGFLOW_TEST_CSV)
print('MTGFlow train:', MTGFLOW_TRAIN_CSV)
print('SDE predictions:', SDE_PREDICTIONS_CSV)
print('Output:', OUT_DIR)

## Protocollo

Il CSV MTGFlow contiene una threshold diversa per ogni località, perché ciascun modello viene calibrato sui propri score storici. Il modo raccomandato mantiene questa calibrazione e varia un moltiplicatore globale `alpha`:

```text
threshold_i(alpha) = alpha × threshold_i_originale
```

L'asse X rappresenta quindi la threshold MTGFlow relativa: `alpha=1` è la classificazione già salvata, valori inferiori classificano più campioni come rari, valori superiori sono più selettivi. La modalità `absolute_score` applica invece la stessa threshold grezza a tutte le località ed è inclusa soltanto per una lettura letterale della richiesta; è meno appropriata quando i modelli MTGFlow sono addestrati separatamente.

In [ ]:
required_paths = [MTGFLOW_TEST_CSV, MTGFLOW_TRAIN_CSV, SDE_PREDICTIONS_CSV]
missing = [path for path in required_paths if not path.is_file()]
if missing:
    candidates = sorted(ROOT.glob('outputs/**/predictions.csv'))
    print('predictions.csv trovati nel repository:')
    for path in candidates[-20:]:
        print(' -', path)
    raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))

mtg_header = set(pd.read_csv(MTGFLOW_TEST_CSV, nrows=0).columns)
train_header = set(pd.read_csv(MTGFLOW_TRAIN_CSV, nrows=0).columns)
pred_header = set(pd.read_csv(SDE_PREDICTIONS_CSV, nrows=0).columns)
assert {'location', 'timestamp', 'anomaly_score', 'threshold'} <= mtg_header
assert {'location', 'anomaly_score'} <= train_header
assert {'location', 'timestamp', 'y_true'} <= pred_header
PREDICTION_COLUMN = 'y_pred_mean' if 'y_pred_mean' in pred_header else 'y_pred'
if PREDICTION_COLUMN not in pred_header:
    raise ValueError('predictions.csv deve contenere y_pred_mean oppure y_pred.')
if DAYTIME_ONLY and 'solar_irradiance_poa_target' not in pred_header:
    raise ValueError('DAYTIME_ONLY richiede solar_irradiance_poa_target.')
print('Colonna di previsione:', PREDICTION_COLUMN)

In [ ]:
mtgflow = pd.read_csv(
    MTGFLOW_TEST_CSV,
    usecols=['location', 'timestamp', 'anomaly_score', 'threshold'],
)
train_scores = pd.read_csv(
    MTGFLOW_TRAIN_CSV, usecols=['location', 'anomaly_score']
)
prediction_columns = ['location', 'timestamp', 'y_true', PREDICTION_COLUMN]
if DAYTIME_ONLY:
    prediction_columns.append('solar_irradiance_poa_target')
predictions = pd.read_csv(SDE_PREDICTIONS_CSV, usecols=prediction_columns)

for frame in (mtgflow, train_scores, predictions):
    frame['location'] = frame['location'].astype(str)
mtgflow['timestamp'] = pd.to_datetime(mtgflow['timestamp'], errors='raise')
predictions['timestamp'] = pd.to_datetime(predictions['timestamp'], errors='raise')
if mtgflow.duplicated(['location', 'timestamp']).any():
    raise ValueError('MTGFlow contiene duplicati location-timestamp.')
if predictions.duplicated(['location', 'timestamp']).any():
    raise ValueError('SDE predictions contiene duplicati location-timestamp.')

joined = predictions.merge(
    mtgflow, on=['location', 'timestamp'], how='inner', validate='one_to_one'
)
coverage = len(joined) / max(len(predictions), 1)
print(f'Predizioni SDE: {len(predictions):,}')
print(f'Righe MTGFlow: {len(mtgflow):,}')
print(f'Righe allineate: {len(joined):,} ({coverage:.2%} delle predizioni)')
if coverage < MIN_JOIN_COVERAGE:
    raise ValueError('Copertura temporale insufficiente: controllare timestamp e score_stride=1.')

numeric = ['y_true', PREDICTION_COLUMN, 'anomaly_score', 'threshold']
joined = joined[np.isfinite(joined[numeric].to_numpy(dtype=float)).all(axis=1)].copy()
if DAYTIME_ONLY:
    joined = joined[joined['solar_irradiance_poa_target'] > DAYTIME_THRESHOLD_WM2].copy()
joined['error'] = joined[PREDICTION_COLUMN] - joined['y_true']
joined['abs_error'] = joined['error'].abs()
joined['squared_error'] = joined['error'].square()
print(f'Campioni analizzati: {len(joined):,} ({"daytime" if DAYTIME_ONLY else "tutti"})')
display(joined.head())

## Costruzione dell'asse delle threshold

Lo sweep viene calcolato ordinando una sola volta i campioni per score relativo. In questo modo non vengono ripetuti decine di filtri su milioni di righe.

In [ ]:
if SWEEP_MODE == 'saved_threshold_multiplier':
    if (joined['threshold'] <= 0).any():
        raise ValueError(
            'Il moltiplicatore richiede threshold positive; usare SWEEP_MODE=absolute_score.'
        )
    coordinate = joined['anomaly_score'].to_numpy(float) / joined['threshold'].to_numpy(float)
    sweep_values = np.asarray(MULTIPLIERS, dtype=float)
    x_label = 'Moltiplicatore threshold MTGFlow (alpha)'
    baseline_coordinate = 1.0
elif SWEEP_MODE == 'absolute_score':
    finite_train = train_scores['anomaly_score'].to_numpy(float)
    finite_train = finite_train[np.isfinite(finite_train)]
    coordinate = joined['anomaly_score'].to_numpy(float)
    sweep_values = np.unique(np.quantile(finite_train, ABSOLUTE_TRAIN_QUANTILES))
    x_label = 'Threshold MTGFlow assoluta'
    baseline_coordinate = float(np.median(joined['threshold']))
else:
    raise ValueError(f'SWEEP_MODE non riconosciuto: {SWEEP_MODE}')

finite = np.isfinite(coordinate)
coordinate = coordinate[finite]
abs_error = joined['abs_error'].to_numpy(float)[finite]
squared_error = joined['squared_error'].to_numpy(float)[finite]
order = np.argsort(coordinate, kind='stable')
sorted_coordinate = coordinate[order]
sorted_abs = abs_error[order]
sorted_squared = squared_error[order]
prefix_abs = np.concatenate(([0.0], np.cumsum(sorted_abs, dtype=np.float64)))
prefix_squared = np.concatenate(([0.0], np.cumsum(sorted_squared, dtype=np.float64)))

In [ ]:
def threshold_sweep(sorted_values, prefix_absolute, prefix_square, thresholds):
    total_n = len(sorted_values)
    total_abs = prefix_absolute[-1]
    total_square = prefix_square[-1]
    rows = []
    for threshold_value in thresholds:
        # normale: coordinate < threshold; raro: coordinate >= threshold
        split = int(np.searchsorted(sorted_values, threshold_value, side='left'))
        n_normal = split
        n_rare = total_n - split
        normal_abs = prefix_absolute[split]
        normal_square = prefix_square[split]
        rare_abs = total_abs - normal_abs
        rare_square = total_square - normal_square
        mae_normal = normal_abs / n_normal if n_normal else np.nan
        mae_rare = rare_abs / n_rare if n_rare else np.nan
        rmse_normal = np.sqrt(normal_square / n_normal) if n_normal else np.nan
        rmse_rare = np.sqrt(rare_square / n_rare) if n_rare else np.nan
        rows.append({
            'threshold_coordinate': float(threshold_value),
            'n_normal': n_normal, 'n_rare': n_rare,
            'rare_fraction': n_rare / total_n if total_n else np.nan,
            'mae_normal': mae_normal, 'mae_rare': mae_rare,
            'rmse_normal': rmse_normal, 'rmse_rare': rmse_rare,
            'mae_gap': mae_rare - mae_normal,
            'rmse_gap': rmse_rare - rmse_normal,
            'mae_ratio': mae_rare / mae_normal if mae_normal else np.nan,
            'rmse_ratio': rmse_rare / rmse_normal if rmse_normal else np.nan,
        })
    return pd.DataFrame(rows)

sweep = threshold_sweep(sorted_coordinate, prefix_abs, prefix_squared, sweep_values)
sweep['mode'] = SWEEP_MODE
sweep['daytime_only'] = DAYTIME_ONLY
sweep.to_csv(OUT_DIR / 'threshold_sweep_metrics.csv', index=False)
display(sweep.head())
display(sweep.iloc[[0, len(sweep)//2, len(sweep)-1]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sweep['threshold_coordinate'], sweep['mae_normal'], label='Normali')
axes[0].plot(sweep['threshold_coordinate'], sweep['mae_rare'], label='Rari/anomali')
axes[0].axvline(baseline_coordinate, color='black', linestyle='--', alpha=0.7, label='Threshold attuale')
axes[0].set(xlabel=x_label, ylabel='MAE [W]', title='MAE in funzione della threshold')
axes[0].grid(alpha=0.25); axes[0].legend()

axes[1].plot(sweep['threshold_coordinate'], sweep['rmse_normal'], label='Normali')
axes[1].plot(sweep['threshold_coordinate'], sweep['rmse_rare'], label='Rari/anomali')
axes[1].axvline(baseline_coordinate, color='black', linestyle='--', alpha=0.7, label='Threshold attuale')
axes[1].set(xlabel=x_label, ylabel='RMSE [W]', title='RMSE in funzione della threshold')
axes[1].grid(alpha=0.25); axes[1].legend()
plt.tight_layout()
fig.savefig(OUT_DIR / 'error_vs_mtgflow_threshold.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sweep['threshold_coordinate'], 100 * sweep['rare_fraction'], color='tab:red')
axes[0].axvline(baseline_coordinate, color='black', linestyle='--', alpha=0.7)
axes[0].set(xlabel=x_label, ylabel='Campioni rari [%]', title='Variazione della classificazione')
axes[0].grid(alpha=0.25)
axes[1].plot(sweep['threshold_coordinate'], sweep['n_normal'], label='Normali')
axes[1].plot(sweep['threshold_coordinate'], sweep['n_rare'], label='Rari/anomali')
axes[1].axvline(baseline_coordinate, color='black', linestyle='--', alpha=0.7)
axes[1].set(xlabel=x_label, ylabel='Numero campioni', title='Dimensione dei due gruppi')
axes[1].grid(alpha=0.25); axes[1].legend()
plt.tight_layout()
fig.savefig(OUT_DIR / 'classification_vs_mtgflow_threshold.png', dpi=180, bbox_inches='tight')
plt.show()

## Scelta della threshold

Non scegliere automaticamente il punto con MAE/RMSE raro massimo: aumentando la threshold il gruppo raro diventa piccolo e instabile. Valutare insieme separazione dell'errore, percentuale rara e numerosità.

`alpha=1` è la scelta a priori già calibrata dagli score storici MTGFlow. Una scelta differente basata sulle curve del 2019 è un'analisi esplorativa e non una validazione indipendente. Per una threshold finale senza leakage, ripetere la selezione su un anno di validation e congelare `SELECTED_COORDINATE` prima della valutazione 2019.

In [ ]:
support = sweep[(sweep['rare_fraction'] >= 0.005) & (sweep['rare_fraction'] <= 0.20)].copy()
print('Candidati con frazione rara tra 0.5% e 20%:')
display(support[[
    'threshold_coordinate', 'rare_fraction', 'n_rare',
    'mae_normal', 'mae_rare', 'mae_ratio',
    'rmse_normal', 'rmse_rare', 'rmse_ratio',
]])
selected_row = sweep.iloc[(sweep['threshold_coordinate'] - SELECTED_COORDINATE).abs().argmin()]
print('Threshold selezionata:')
display(selected_row.to_frame('value'))

In [ ]:
if SWEEP_MODE == 'saved_threshold_multiplier':
    location_thresholds = (
        mtgflow[['location', 'threshold']].drop_duplicates('location').copy()
    )
    location_thresholds['selected_multiplier'] = float(SELECTED_COORDINATE)
    location_thresholds['selected_threshold'] = (
        location_thresholds['threshold'] * SELECTED_COORDINATE
    )
else:
    location_thresholds = pd.DataFrame({
        'location': sorted(mtgflow['location'].unique()),
        'selected_threshold': float(SELECTED_COORDINATE),
    })
location_thresholds.to_csv(OUT_DIR / 'selected_thresholds_by_location.csv', index=False)
summary = {
    'post_processing_only': True,
    'mtgflow_training_rerun': False,
    'sdenet_training_rerun': False,
    'sweep_mode': SWEEP_MODE,
    'selected_coordinate': float(SELECTED_COORDINATE),
    'daytime_only': DAYTIME_ONLY,
    'join_coverage': float(coverage),
    'n_analyzed': int(len(joined)),
    'selection_warning': 'Use a validation year for unbiased final threshold selection.',
}
(OUT_DIR / 'threshold_analysis_metadata.json').write_text(
    json.dumps(summary, indent=2), encoding='utf-8'
)
print('File prodotti:')
for path in sorted(OUT_DIR.iterdir()):
    print(' -', path.name)